# GazeSpeakZero — E2: how well does each method name the useful objects in an everyday scene?

**Before you start: Runtime → Change runtime type → T4 GPU → Save.** (A GPU is needed for the vision models.)

Run the cells **one at a time** with Shift + Enter. Total time is about 1–1.5 hours, most of it unattended. If the install cell prints a warning about restarting the session, do Runtime → Restart session and run from the top again. Everything is saved to Google Drive (`MyDrive/gazespeakzero/E2`), so if Colab disconnects, run the cells again and it resumes where it stopped.

What it does: 300 everyday indoor photos (bathroom, bedroom, kitchen, living room, dining) from LVIS/COCO. Four methods each propose 12 items per photo, and their items are checked against the photos' object labels:
- a fixed list with no camera (the Look to Speak style);
- CLIP;
- the original paper's BLIP-2 step;
- a small modern vision-language model, Qwen2.5-VL-3B.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/gazespeakzero/E2'
import os, torch; os.makedirs(WORK, exist_ok=True); print(WORK)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to T4 GPU before continuing')
!df -h /content | tail -1

In [ ]:
!pip -q install "transformers==4.51.3" "huggingface_hub<1.0" "tokenizers<0.22" "accelerate>=0.30" requests pandas
import transformers
from transformers import CLIPModel, Blip2ForConditionalGeneration, Qwen2_5_VLForConditionalGeneration
print('transformers', transformers.__version__, '- imports OK')

## 1. Annotations (about 5 minutes)

In [ ]:
%%bash
cd /content
[ -f lvis_v1_val.json ] || { wget -q --show-progress -O lvis_v1_val.json.zip https://dl.fbaipublicfiles.com/LVIS/lvis_v1_val.json.zip && unzip -q -o lvis_v1_val.json.zip && rm lvis_v1_val.json.zip; }
[ -f instances_val2017.json ] || { wget -q --show-progress -O ann.zip http://images.cocodataset.org/annotations/annotations_trainval2017.zip && unzip -q -o -j ann.zip annotations/instances_train2017.json annotations/instances_val2017.json && rm ann.zip; }
[ -f coco_to_synset.json ] || wget -q -O coco_to_synset.json https://raw.githubusercontent.com/lvis-dataset/lvis-api/master/data/coco_to_synset.json
ls -la lvis_v1_val.json instances_train2017.json instances_val2017.json coco_to_synset.json

In [ ]:
!mkdir -p /content/e2

In [ ]:
%%writefile /content/e2/aac_vocab.py
"""
E2: the AAC-relevant vocabulary and the text-to-LVIS matcher. Fixed before any
E2 result was seen.

AAC_VOCAB lists everyday objects that a person with severe motor impairment at
home or in a care setting might ask for, use or talk about: drink and food,
personal care and medical items, comfort and bedding, devices, furniture, and
pets. The entries are LVIS category names. Entries that the LVIS version in use
does not contain are reported and dropped, never silently remapped.
"""
from __future__ import annotations

import re

AAC_VOCAB = [
    # drink and food
    "bottle", "water_bottle", "cup", "mug", "glass_(drink_container)", "wineglass", "kettle", "teapot",
    "coffee_maker", "milk", "orange_juice", "banana", "apple", "orange_(fruit)", "bread", "sandwich",
    "cake", "cookie", "doughnut", "pizza", "bowl", "plate", "spoon", "fork", "knife", "straw_(for_drinking)",
    "napkin",
    # personal care and medical
    "toothbrush", "toothpaste", "hairbrush", "comb", "towel", "hand_towel", "soap", "toilet_tissue",
    "tissue_paper", "medicine", "pill", "spectacles", "wheelchair", "crutch", "walking_stick",
    "toilet", "sink", "bathtub", "shower_head",
    # comfort, bedding, clothing
    "bed", "pillow", "cushion", "blanket", "quilt", "bedspread", "slipper_(footwear)", "sock", "shoe",
    "jacket", "sweater", "hat",
    # environment and devices
    "lamp", "table_lamp", "fan", "heater", "air_conditioner", "curtain", "clock", "wall_clock",
    "alarm_clock", "television_set", "remote_control", "radio_receiver", "cellular_telephone",
    "telephone", "laptop_computer", "book", "newspaper", "magazine", "pen", "notebook", "headset",
    "earphone",
    # furniture
    "chair", "armchair", "recliner", "sofa", "table", "dining_table", "coffee_table", "desk",
    "refrigerator", "microwave_oven", "stove", "trash_can",
    # other
    "dog", "cat", "vase", "painting", "mirror", "flower_arrangement",
]

ROOM_INDICATORS = [          # priority order: the first matching room wins
    ("bathroom", ["toilet", "bathtub", "shower_curtain", "shower_head"]),
    ("bedroom", ["bed", "bunk_bed"]),
    ("kitchen", ["refrigerator", "stove", "microwave_oven", "oven", "kitchen_sink", "dishwasher"]),
    ("living room", ["sofa", "armchair", "television_set", "coffee_table"]),
    ("dining", ["dining_table"]),
]

STOP = {"a", "an", "the", "some", "of", "with", "on", "in", "and", "or", "for", "to", "at", "by"}
CUT = {"of", "for", "on", "with", "in", "at", "from", "near"}   # 'glass of water' -> 'glass'

# Everyday words -> LVIS names, fixed before any output was seen. Applied to every
# method alike. A target missing from the LVIS release in use is skipped.
ALIASES = {
    "fridge": "refrigerator", "laptop": "laptop_computer", "microwave": "microwave_oven",
    "tv": "television_set", "television": "television_set", "telly": "television_set",
    "remote": "remote_control", "tv remote": "remote_control",
    "phone": "cellular_telephone", "cell phone": "cellular_telephone", "cellphone": "cellular_telephone",
    "mobile phone": "cellular_telephone", "mobile": "cellular_telephone", "smartphone": "cellular_telephone",
    "wine glass": "wineglass", "glasses": "spectacles", "eyeglasses": "spectacles",
    "couch": "sofa", "settee": "sofa", "bin": "trash_can", "dustbin": "trash_can", "garbage can": "trash_can",
    "toilet paper": "toilet_tissue", "tissues": "tissue_paper", "tissue": "tissue_paper",
    "water glass": "glass_(drink_container)", "drinking glass": "glass_(drink_container)",
    "mouse": "mouse_(computer_equipment)", "computer mouse": "mouse_(computer_equipment)",
    "tap": "faucet", "duvet": "quilt", "comforter": "quilt", "bedsheet": "sheet_(bedding)",
    "potted plant": "flowerpot", "plant pot": "flowerpot", "teddy": "teddy_bear",
}

# Lenient scoring: an item also counts as correct if a category in the same group is present.
LENIENT_GROUPS = [
    {"table", "dining_table", "coffee_table", "kitchen_table", "desk"},
    {"clock", "wall_clock", "alarm_clock"},
    {"lamp", "table_lamp"},
    {"towel", "hand_towel", "bath_towel", "dishtowel", "paper_towel"},
    {"blanket", "quilt", "bedspread"},
    {"cup", "mug"},
    {"glass_(drink_container)", "wineglass"},
    {"telephone", "cellular_telephone"},
    {"bottle", "water_bottle", "beer_bottle", "wine_bottle"},
    {"pillow", "cushion"},
]


def canon(name: str) -> str:
    """LVIS name -> plain words: 'glass_(drink_container)' -> 'glass', 'television_set' -> 'television set'."""
    name = re.sub(r"_?\(.*?\)", "", name)
    return name.replace("_", " ").strip().lower()


def _singular_candidates(w: str) -> list[str]:
    out = []
    for suf, rep in (("ies", "y"), ("ves", "f"), ("ves", "fe"), ("oes", "o"), ("ches", "ch"),
                     ("shes", "sh"), ("sses", "ss"), ("xes", "x"), ("es", ""), ("s", "")):
        if w.endswith(suf) and len(w) > len(suf) + 1:
            out.append(w[: -len(suf)] + rep)
    return out


class Matcher:
    """Maps free text ('Two white pillows') to one LVIS category id, or None.

    Tries, in order: the whole phrase, the phrase singularised, then shorter
    trailing phrases (so 'wooden coffee table' -> 'coffee table' -> 'table'),
    always by exact equality with a canonicalised name or synonym."""

    def __init__(self, categories: list[dict]):
        self.id2name = {c["id"]: c["name"] for c in categories}
        by_name = {c["name"]: c["id"] for c in categories}
        self.lookup: dict[str, int] = {}
        self.clashes: dict[str, list[str]] = {}
        for c in sorted(categories, key=lambda c: c["id"]):
            for s in [c["name"]] + list(c.get("synonyms", [])):
                k = canon(s)
                if k in self.lookup and self.lookup[k] != c["id"]:
                    self.clashes.setdefault(k, [self.id2name[self.lookup[k]]]).append(c["name"])
                self.lookup.setdefault(k, c["id"])          # first (lowest id) wins on a clash
        own = {}                                            # a category's own name beats others' synonyms
        for c in sorted(categories, key=lambda c: c["id"]):
            own.setdefault(canon(c["name"]), c["id"])       # name-vs-name clash: lowest id
        self.lookup.update(own)
        self.aliases_used = {}
        for k, target in ALIASES.items():
            if target in by_name:
                self.lookup[k] = by_name[target]
                self.aliases_used[k] = target
        self.group = {}
        for g in LENIENT_GROUPS:
            ids = {by_name[n] for n in g if n in by_name}
            for i in ids:
                self.group[i] = ids
        self.head = {cid: canon(n).split()[-1] for cid, n in self.id2name.items()}

    def match(self, text: str):
        t = re.sub(r"[^a-z\s-]", " ", text.lower()).replace("-", " ")
        raw = t.split()
        whole = " ".join(w for w in raw if w not in {"a", "an", "the"})
        if whole in self.lookup:                         # 'chest of drawers' as a whole name
            return self.lookup[whole]
        for j, w in enumerate(raw):                      # 'glass of water' -> 'glass'
            if w in CUT and j > 0:
                raw = raw[:j]
                break
        words = [w for w in raw if w not in STOP]
        if not words:
            return None
        for i in range(len(words)):                      # longest trailing phrase first
            phrase = words[i:]
            cands = [" ".join(phrase)] + [" ".join(phrase[:-1] + [w]) for w in _singular_candidates(phrase[-1])]
            for cand in cands:
                if cand in self.lookup:
                    return self.lookup[cand]
        return None

    def resolve_vocab(self, names: list[str]):
        ids, missing = [], []
        by_name = {v: k for k, v in self.id2name.items()}
        for n in names:
            cid = by_name.get(n)                               # exact LVIS names only
            if cid is None:
                missing.append(n)
            elif cid not in ids:
                ids.append(cid)
        return ids, missing


In [ ]:
%%writefile /content/e2/e2_build_set.py
"""
E2, step 1: choose the everyday indoor scenes and write their ground truth.

    python e2_build_set.py --lvis lvis_v1_val.json --out e2_set --per-room 60 \
        --coco-ann instances_train2017.json instances_val2017.json --coco-synset coco_to_synset.json

The scenes come from the LVIS v1 validation set (COCO 2017 photos annotated with
more than 1,200 object categories). An image qualifies if:
  * it contains a room indicator (see ROOM_INDICATORS), which also gives its room;
  * at least 3 AAC-vocabulary categories are present in it.
Up to --per-room images are drawn per room with a fixed seed.

Writes set.jsonl (one line per image: id, url, room, positive category ids,
verified-absent category ids) and vocab.json, and downloads the images to
out/images/. Nothing here depends on any method's output.
"""
from __future__ import annotations

import argparse
import collections
import json
import os
import random
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
from aac_vocab import AAC_VOCAB, ROOM_INDICATORS, Matcher  # noqa: E402

SEED = 0
# COCO "dining table" covers any table, "tv" includes monitors, "cup" includes mugs and glasses:
# their PRESENCE is not taken as LVIS presence (their absence still is).
COCO_NO_POSITIVE = {"dining table", "tv", "cup"}
# Small objects COCO often leaves unlabelled: their ABSENCE is not trusted.
COCO_NO_NEGATIVE = {"spoon", "knife", "fork", "remote", "toothbrush", "cell phone", "scissors", "mouse",
                    "book", "hair drier"}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--lvis", required=True)
    ap.add_argument("--out", default="e2_set")
    ap.add_argument("--per-room", type=int, default=60)
    ap.add_argument("--min-aac", type=int, default=3)
    ap.add_argument("--no-download", action="store_true")
    ap.add_argument("--coco-ann", nargs="*", default=[],
                    help="COCO 2017 instances_*.json. COCO labels its 80 classes exhaustively, so a COCO class "
                         "absent from a photo is verified absent; mapped to LVIS via coco_to_synset.json")
    ap.add_argument("--coco-synset", default=None, help="coco_to_synset.json from the lvis-api repository")
    a = ap.parse_args()
    os.makedirs(os.path.join(a.out, "images"), exist_ok=True)

    d = json.load(open(a.lvis))
    m = Matcher(d["categories"])
    vocab_ids, missing = m.resolve_vocab(AAC_VOCAB)
    print(f"AAC vocabulary: {len(vocab_ids)} of {len(AAC_VOCAB)} names found in this LVIS release")
    if missing:
        print("  not in LVIS (dropped):", ", ".join(missing))
    room_ids = []
    for room, names in ROOM_INDICATORS:
        ids, miss = m.resolve_vocab(names)
        room_ids.append((room, set(ids)))
        if miss:
            print(f"  room indicator names not in LVIS for {room}: {miss}")

    pos = collections.defaultdict(set)
    for ann in d["annotations"]:
        pos[ann["image_id"]].add(ann["category_id"])
    del d["annotations"]                         # free memory before the COCO files are read
    import gc; gc.collect()
    vset = set(vocab_ids)
    by_room = collections.defaultdict(list)
    for img in d["images"]:
        p = pos.get(img["id"], set())
        room = next((r for r, ids in room_ids if p & ids), None)
        if room is None or len(p & vset) < a.min_aac:
            continue
        by_room[room].append(img)
    rng = random.Random(SEED)
    chosen = []
    for room, _ in ROOM_INDICATORS:
        imgs = sorted(by_room[room], key=lambda i: i["id"])
        rng.shuffle(imgs)
        chosen += [(room, i) for i in imgs[: a.per_room]]
        print(f"  {room}: {len(by_room[room])} eligible, {min(len(imgs), a.per_room)} chosen")

    # COCO exhaustive labels -> extra verified-present / verified-absent LVIS categories
    coco_pos, coco_known = {}, set()
    coco_map, coco_neg_ok, coco_pos_ok = {}, set(), set()
    coco_n_ann = collections.Counter()
    if a.coco_ann and a.coco_synset:
        syn = json.load(open(a.coco_synset))
        lvis_by_synset = {c.get("synset"): c["id"] for c in d["categories"] if c.get("synset")}
        want = {img["id"] for _, img in chosen}
        for path in a.coco_ann:
            cj = json.load(open(path))
            cname = {c["id"]: c["name"] for c in cj["categories"]}
            for c in cj["categories"]:
                lid = lvis_by_synset.get(syn.get(c["name"], {}).get("synset"))
                if lid is not None:
                    coco_map[c["id"]] = lid
                    if c["name"] not in COCO_NO_NEGATIVE:
                        coco_neg_ok.add(lid)
                    if c["name"] not in COCO_NO_POSITIVE:
                        coco_pos_ok.add(lid)
            for im in cj["images"]:
                if im["id"] in want:
                    coco_known.add(im["id"])
                    coco_pos.setdefault(im["id"], set())
            for an in cj["annotations"]:
                if an["image_id"] in want:
                    coco_n_ann[an["image_id"]] += 1
                    if an["category_id"] in coco_map:
                        coco_pos[an["image_id"]].add(coco_map[an["category_id"]])
            del cj
        coco_known = {i for i in coco_known if coco_n_ann[i] > 0}   # trust absences only for labelled images
        print(f"COCO: {len(coco_map)} of 80 classes map to LVIS; {len(coco_known)} of {len(want)} scenes found "
              f"in COCO files with at least one label")
        print("  COCO -> LVIS:", ", ".join(sorted(f"{cname_} -> {d_}" for cname_, d_ in
              ((n, next((c['name'] for c in d['categories'] if c['id'] == coco_map[i]), '?'))
               for i, n in cname.items() if i in coco_map)))[:2000])

    rows = []
    for room, img in chosen:
        url = img.get("coco_url") or f"http://images.cocodataset.org/val2017/{img['id']:012d}.jpg"
        rows.append({"image_id": img["id"], "url": url, "room": room, "width": img.get("width"),
                     "height": img.get("height"), "pos": sorted(pos[img["id"]]),
                     "neg": sorted(img.get("neg_category_ids", [])),
                     # extended with COCO exhaustive labels (primary for scoring)
                     "pos_x": sorted(pos[img["id"]] | ((coco_pos.get(img["id"], set()) & coco_pos_ok)
                                                       - set(img.get("neg_category_ids", [])))),
                     "neg_x": sorted((set(img.get("neg_category_ids", [])) |
                                      ((coco_neg_ok - coco_pos.get(img["id"], set())) if img["id"] in coco_known else set()))
                                     - pos[img["id"]] - coco_pos.get(img["id"], set())),
                     "not_exhaustive": sorted(img.get("not_exhaustive_category_ids", []))})
    with open(os.path.join(a.out, "set.jsonl"), "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    json.dump({"vocab_ids": vocab_ids, "vocab_names": [m.id2name[i] for i in vocab_ids],
               "missing": missing, "categories": d["categories"]},
              open(os.path.join(a.out, "vocab.json"), "w"))
    print(f"{len(rows)} scenes written")

    if a.no_download:
        return
    import requests
    ok = 0
    for r in rows:
        dst = os.path.join(a.out, "images", f"{r['image_id']}.jpg")
        if os.path.exists(dst) and os.path.getsize(dst) > 1000:
            ok += 1
            continue
        for attempt in range(3):
            try:
                resp = requests.get(r["url"], timeout=30)
                resp.raise_for_status()
                open(dst, "wb").write(resp.content)
                ok += 1
                break
            except Exception as e:
                if attempt == 2:
                    print(f"  download failed {r['image_id']}: {e}")
                time.sleep(2)
    print(f"images downloaded: {ok}/{len(rows)}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/e2/e2_generate.py
"""
E2, step 2: each method proposes a ranked list of up to 12 items for every scene.

    python e2_generate.py --set e2_set --out e2_outputs --methods static clip blip2_orig qwen_vl [--limit 5]

Methods
-------
static      No vision. The AAC-vocabulary items ranked by how often they occur
            in the OTHER scenes of the set (leave-one-out), using the same ground truth
            as the scoring. This is the best
            possible fixed, pre-written list, as Look to Speak would use, built
            with hindsight, so it is a generous baseline.
clip        CLIP ViT-L/14 (the original paper's scene model), used as it
            should be used: ranks every AAC-vocabulary item by image-text
            similarity to "a photo of a {item}." Closed vocabulary.
blip2_orig  The original GazeSpeakZero Stage 2, verbatim: BLIP-2 (OPT-2.7B)
            answers "What are the objects and areas visible in this scene?",
            the decoded text is split into words, stopwords and words of two
            letters or fewer are removed, and the first 12 unique words are kept.
blip2_caption  Fair BLIP-2 baseline 1: the same model with no text prompt, i.e. its
            native captioning mode (beam search, 3 beams). The caption is cut into
            noun phrases at commas, "and" and spatial words ("a kitchen with a
            stove and a sink" -> kitchen, stove, sink).
blip2_qa    Fair BLIP-2 baseline 2: the prompt format BLIP-2 was trained on,
            "Question: What objects are in this room? Answer:", same phrase parsing.
            Both variants strip the echoed prompt from the decoded text (under
            transformers 4.51 the decoded output starts with the prompt; in E2
            the verbatim original step returned only the echo for all 283 scenes).
qwen_vl     Qwen2.5-VL-3B-Instruct, a small open vision-language model, asked
            for a ranked comma-separated list (prompt below). Open vocabulary.

Output: one JSON line per (method, image) with the ranked items and the seconds
the method took for that image (GPU time where a GPU is used). Resumable: a
(method, image) pair already in the output file is skipped.
"""
from __future__ import annotations

import argparse
import collections
import json
import os
import re
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
from aac_vocab import canon  # noqa: E402

K_MAX = 12
QWEN_PROMPT = ("List the objects in this scene that a person here might ask for, use or talk about. "
               "Reply with only a comma-separated list of up to 12 short object names, most useful first.")
BLIP_PROMPT = "What are the objects and areas visible in this scene?"          # original, verbatim
BLIP_STOPWORDS = {                                                               # original, verbatim
    "a", "an", "the", "is", "are", "and", "of", "in", "on", "with", "this",
    "that", "to", "for", "at", "it", "its", "there", "scene", "image",
    "visible", "objects", "areas", "what",
}


BLIP_QA_PROMPT = "Question: What objects are in this room? Answer:"
_ING_NOUNS = ("ceiling|painting|paintings|clothing|building|buildings|ring|rings|string|wing|wings|bedding|"
              "icing|frosting|pudding|dressing|railing|lighting|seating|flooring|awning|earring|earrings|"
              "stuffing|wedding|spring|swing|king|thing|things|something|nothing|everything|ironing board|"
              "sling|filling|topping|toppings|siding|molding|moulding|outing|living|dining|sewing|washing|baking|cutting|"
              "shopping|rolling|frying|sleeping|running|walking|hearing|reading|drinking|changing|sitting room|"
              "dressing table|mixing|serving|measuring|boxing|parking|ping")
# split at punctuation, conjunctions, prepositions (longest first) and any "-ing" verb that is not a noun above
_SPLIT = re.compile(r"[,;.:!?\n]|\b(?:in front of|on top of|next to|there is|there are|and|with|on|in|at|near|"
                    r"beside|under|behind|by|including|is|are|has|have|their|his|her|its)\b"
                    r"|\b(?!(?:" + _ING_NOUNS + r")\b)[a-z]+ing\b")
_LEAD = re.compile(r"^(?:a|an|the|some|two|three|four|five|several|many|other|various|small|large|"
                   r"big|white|black|red|blue|green|wooden|old|\d+)\s+")


def phrase_items(text: str) -> list[str]:
    """Noun phrases from a BLIP-2 caption or answer, in order, without repeats."""
    items = []
    for ph in _SPLIT.split(text.lower()):
        ph = re.sub(r"[^a-z\s-]", " ", ph or "").strip()
        prev = None
        while ph and ph != prev:                  # strip leading articles / numbers / colours
            prev, ph = ph, _LEAD.sub("", ph).strip()
        if ph and len(ph) >= 2 and len(ph.split()) <= 4 and ph not in items and ph not in BLIP_STOPWORDS:
            items.append(ph)
    return items[:K_MAX]


def strip_prompt(decoded: str, prompt: str) -> str:
    d = decoded.strip()
    return d[len(prompt):].strip() if prompt and d.lower().startswith(prompt.lower()) else d


def load_set(d):
    rows = [json.loads(l) for l in open(os.path.join(d, "set.jsonl"))]
    vocab = json.load(open(os.path.join(d, "vocab.json")))
    rows = [r for r in rows if os.path.exists(os.path.join(d, "images", f"{r['image_id']}.jpg"))]
    return rows, vocab


def parse_list(text: str) -> list[str]:
    text = text.replace("\n", ",").replace(";", ",")
    items = []
    for t in text.split(","):
        t = re.sub(r"^\s*(\d+[\.\)]|[-*•])\s*", "", t).strip().strip(".").strip()
        if t and len(t) <= 40 and t.lower() not in [i.lower() for i in items]:
            items.append(t)
    return items[:K_MAX]


def run_static(rows, vocab, emit):
    vids = vocab["vocab_ids"]
    names = {i: canon(n) for i, n in zip(vids, vocab["vocab_names"])}
    count = collections.Counter()
    for r in rows:
        count.update(set(r.get("pos_x", r["pos"])) & set(vids))
    for r in rows:
        own = set(r.get("pos_x", r["pos"])) & set(vids)
        loo = {i: count[i] - (1 if i in own else 0) for i in vids}      # leave this image out
        ranked = sorted(vids, key=lambda i: (-loo[i], names[i]))[:K_MAX]
        emit("static", r, [names[i] for i in ranked], 0.0)


def run_clip(rows, d, vocab, emit, device):
    import torch
    from PIL import Image
    from transformers import CLIPModel, CLIPProcessor
    model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(device).eval()
    proc = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
    names = [canon(n) for n in vocab["vocab_names"]]
    with torch.no_grad():
        t = proc(text=[f"a photo of a {n}." for n in names], return_tensors="pt", padding=True).to(device)
        tf = model.get_text_features(**t)
        tf = tf / tf.norm(dim=-1, keepdim=True)
        for r in rows:
            img = Image.open(os.path.join(d, "images", f"{r['image_id']}.jpg")).convert("RGB")
            t0 = time.time()
            x = proc(images=img, return_tensors="pt").to(device)
            f = model.get_image_features(**x)
            f = f / f.norm(dim=-1, keepdim=True)
            sims = (f @ tf.T)[0].float().cpu().numpy()
            order = sims.argsort()[::-1][:K_MAX]
            if device == "cuda":
                torch.cuda.synchronize()
            emit("clip", r, [names[i] for i in order], time.time() - t0)
    del model
    _free()


def run_blip2(rows, d, emit, device):
    import torch
    from PIL import Image
    from transformers import Blip2ForConditionalGeneration, Blip2Processor
    dtype = torch.float16 if device == "cuda" else torch.float32
    proc = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
    model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-2.7b", torch_dtype=dtype).to(device).eval()
    for r in rows:
        img = Image.open(os.path.join(d, "images", f"{r['image_id']}.jpg")).convert("RGB")
        t0 = time.time()
        x = proc(images=img, text=BLIP_PROMPT, return_tensors="pt").to(device, dtype)
        with torch.no_grad():
            out = model.generate(**x, max_new_tokens=40)
        caption = proc.batch_decode(out, skip_special_tokens=True)[0]
        toks = [t for t in re.findall(r"[a-zA-Z]+", caption.lower()) if t not in BLIP_STOPWORDS and len(t) > 2]
        labels = list(dict.fromkeys(toks))[:K_MAX] or ["object", "person", "area"]   # original fallback
        emit("blip2_orig", r, labels, time.time() - t0, raw=caption)
    del model
    _free()


def run_blip2_fair(rows, d, emit, device, variant):
    import torch
    from PIL import Image
    from transformers import Blip2ForConditionalGeneration, Blip2Processor
    dtype = torch.float16 if device == "cuda" else torch.float32
    proc = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
    model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-2.7b", torch_dtype=dtype).to(device).eval()
    prompt = BLIP_QA_PROMPT if variant == "blip2_qa" else None
    empty = 0
    for i, r in enumerate(rows):
        img = Image.open(os.path.join(d, "images", f"{r['image_id']}.jpg")).convert("RGB")
        t0 = time.time()
        x = (proc(images=img, text=prompt, return_tensors="pt") if prompt else proc(images=img, return_tensors="pt"))
        x = x.to(device, dtype)
        with torch.no_grad():
            out = model.generate(**x, max_new_tokens=40, num_beams=3, repetition_penalty=1.3)
        if device == "cuda":
            torch.cuda.synchronize()
        raw = strip_prompt(proc.batch_decode(out, skip_special_tokens=True)[0], prompt or "")
        items = phrase_items(raw)
        empty += not items
        if i < 3:
            print(f"  {variant} raw: {raw!r} -> {items}", flush=True)
        emit(variant, r, items, time.time() - t0, raw=raw)
    print(f"  {variant}: {empty} of {len(rows)} scenes gave no items", flush=True)
    del model
    _free()


def run_qwen(rows, d, emit, device):
    import torch
    from PIL import Image
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    mid = "Qwen/Qwen2.5-VL-3B-Instruct"
    proc = AutoProcessor.from_pretrained(mid, min_pixels=256 * 28 * 28, max_pixels=640 * 28 * 28)

    def load(dtype):
        return Qwen2_5_VLForConditionalGeneration.from_pretrained(mid, torch_dtype=dtype, attn_implementation="sdpa").to(device).eval()

    def ask(model, img):
        msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": QWEN_PROMPT}]}]
        text = proc.apply_chat_template(msgs, add_generation_prompt=True)
        x = proc(text=[text], images=[img], return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(**x, max_new_tokens=80, do_sample=False)
        return proc.batch_decode(out[:, x["input_ids"].shape[1]:], skip_special_tokens=True)[0]

    # Qwen2.5-VL can overflow in fp16. Probe 3 scenes; if fewer than 2 give a usable list,
    # reload in bf16 (emulated on a T4: slower, but it fits; fp32 would not fit in 15 GB).
    dtype = torch.float16 if device == "cuda" else torch.float32
    model = load(dtype)
    probes = [Image.open(os.path.join(d, "images", f"{r['image_id']}.jpg")).convert("RGB") for r in rows[:3]]
    answers = [ask(model, im) for im in probes]
    good = sum(len(parse_list(x)) >= 2 for x in answers)
    if dtype == torch.float16 and good < min(2, len(probes)):
        print(f"  qwen fp16 answers unusable ({answers[0][:60]!r}); reloading in bfloat16", flush=True)
        del model
        _free()
        dtype = torch.bfloat16
        model = load(dtype)
        answers = [ask(model, im) for im in probes]
    print(f"  qwen dtype {dtype}; first answer: {answers[0][:120]!r}", flush=True)
    short = 0
    for r in rows:
        img = Image.open(os.path.join(d, "images", f"{r['image_id']}.jpg")).convert("RGB")
        t0 = time.time()
        raw = ask(model, img)
        if device == "cuda":
            torch.cuda.synchronize()
        items = parse_list(raw)
        short += len(items) < 2
        emit("qwen_vl", r, items, time.time() - t0, raw=raw)
    print(f"  qwen: {short} of {len(rows)} scenes gave fewer than 2 items", flush=True)
    del model
    _free()


def _free():
    import gc
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--set", default="e2_set")
    ap.add_argument("--out", default="e2_outputs")
    ap.add_argument("--methods", nargs="+", default=["static", "clip", "blip2_orig", "qwen_vl"])
    ap.add_argument("--limit", type=int, default=None, help="dry run: first N scenes only")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    rows, vocab = load_set(a.set)
    if a.limit:                                  # dry run: spread the scenes over the rooms
        by = collections.defaultdict(list)
        for r in rows:
            by[r["room"]].append(r)
        picked = []
        while len(picked) < min(a.limit, len(rows)):
            for room in list(by):
                if by[room] and len(picked) < a.limit:
                    picked.append(by[room].pop(0))
        rows = picked
    path = os.path.join(a.out, "outputs.dry.jsonl" if a.limit else "outputs.jsonl")
    done = set()
    if os.path.exists(path):                    # drop a half-written last line left by a disconnect
        good = []
        for l in open(path):
            try:
                o = json.loads(l)
                done.add((o["method"], o["image_id"]))
                good.append(l if l.endswith("\n") else l + "\n")
            except json.JSONDecodeError:
                pass
        with open(path, "w") as g:
            g.writelines(good)
    try:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
    except ImportError:
        device = "cpu"
    print(f"{len(rows)} scenes | device {device} | methods {a.methods}", flush=True)
    f = open(path, "a")
    n = collections.Counter()

    def emit(method, r, items, secs, raw=None):
        f.write(json.dumps({"method": method, "image_id": r["image_id"], "items": items[:K_MAX],
                            "seconds": round(secs, 4), "raw": raw}) + "\n")
        f.flush()
        n[method] += 1
        if n[method] % 50 == 0:
            print(f"  {method}: {n[method]} scenes", flush=True)

    for m in a.methods:
        todo = [r for r in rows if (m, r["image_id"]) not in done]
        if not todo:
            print(f"{m}: already done")
            continue
        print(f"{m}: {len(todo)} scenes ...", flush=True)
        t0 = time.time()
        if m == "static":
            run_static(rows, vocab, lambda mm, r, it, s, raw=None: emit(mm, r, it, s, raw)
                       if (mm, r["image_id"]) not in done else None)
        elif m == "clip":
            run_clip(todo, a.set, vocab, emit, device)
        elif m == "blip2_orig":
            run_blip2(todo, a.set, emit, device)
        elif m in ("blip2_caption", "blip2_qa"):
            run_blip2_fair(todo, a.set, emit, device, m)
        elif m == "qwen_vl":
            run_qwen(todo, a.set, emit, device)
        else:
            raise SystemExit(f"unknown method {m}")
        print(f"{m}: done in {time.time() - t0:.0f} s", flush=True)
    f.close()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/e2/e2_score.py
"""
E2, step 3: score every method's ranked items against LVIS ground truth.

    python e2_score.py --set e2_set --outputs e2_outputs/outputs.jsonl --out e2_results

Matching. Each proposed item is mapped to at most one LVIS category by
aac_vocab.Matcher, which uses LVIS names and synonyms plus a fixed alias table
('fridge', 'tv', 'cell phone', ...). The same rules apply to every method. If a
method names the same category twice, only the first is kept.

Ground truth. LVIS is federated: per image, some categories are verified present
(pos), some verified absent (neg), and the rest are unknown. The primary scoring
extends both sets with COCO's exhaustive labels for its 80 classes (pos_x, neg_x;
see e2_build_set.py for the safeguards). LVIS-only figures are a pre-stated
check. Each item is:

    correct        its category is verified present
    wrong          its category is verified absent
    unverifiable   no LVIS category, or not checked in this photo

Lenient: an item also counts as correct if a category in the same small
equivalence group is present (aac_vocab.LENIENT_GROUPS, e.g. table ~
dining_table).

Per image, at K = 3, 6, 12 (items are shown 3 at a time, so K = 3 is one screen):
  precision@K   correct / (correct + wrong) among the top K; NaN if none verifiable
  verifiable@K  (correct + wrong) / K; an empty or short list counts against it
  recall@K      share of AAC-vocabulary objects present in the photo named in the top K
  hit@3         at least one strictly correct item on the first screen
                (hit_lenient@3 is also reported)

Only scenes that every method covered are scored. Means are over scenes, with
95% bootstrap CIs. Differences between methods use a paired bootstrap over the
same resampled scenes. Pooled precision (all items together) is reported as
well. Also writes a blinded audit sheet.
"""
from __future__ import annotations

import argparse
import json
import os
import random
import sys

import numpy as np
import pandas as pd

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
from aac_vocab import Matcher  # noqa: E402

KS = [3, 6, 12]
N_BOOT = 5000
SEED = 0
ORDER = ["static", "clip", "blip2_orig", "blip2_caption", "blip2_qa", "qwen_vl"]
PAIRS = [("qwen_vl", "static"), ("qwen_vl", "clip"), ("clip", "static"), ("qwen_vl", "blip2_orig"), ("clip", "blip2_orig"),
         ("blip2_caption", "static"), ("blip2_qa", "static"), ("qwen_vl", "blip2_caption"), ("qwen_vl", "blip2_qa"),
         ("clip", "blip2_caption"), ("clip", "blip2_qa")]


def boot(v, rng):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if len(v) < 2:
        return (np.nan, np.nan)
    m = v[rng.integers(0, len(v), (N_BOOT, len(v)))].mean(1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))


def paired(a, b, rng):
    """mean(a - b) over scenes where both are finite, with a bootstrap CI."""
    ok = np.isfinite(a) & np.isfinite(b)
    d = (a - b)[ok]
    if len(d) < 2:
        return np.nan, (np.nan, np.nan), int(ok.sum())
    m = d[rng.integers(0, len(d), (N_BOOT, len(d)))].mean(1)
    return float(d.mean()), (float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))), int(ok.sum())


def norm_item(s: str) -> str:
    return " ".join("".join(ch for ch in s.lower() if ch.isalnum() or ch == " ").split())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--set", default="e2_set")
    ap.add_argument("--outputs", default="e2_outputs/outputs.jsonl")
    ap.add_argument("--out", default="e2_results")
    ap.add_argument("--audit-n", type=int, default=50)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    rng = np.random.default_rng(SEED)

    scenes = {r["image_id"]: r for r in (json.loads(l) for l in open(os.path.join(a.set, "set.jsonl")))}
    vocab = json.load(open(os.path.join(a.set, "vocab.json")))
    m = Matcher(vocab["categories"])
    vset = set(vocab["vocab_ids"])

    outs = []
    for l in open(a.outputs):
        try:
            outs.append(json.loads(l))
        except json.JSONDecodeError:
            pass
    # keep the last record per (method, image); score only scenes every method covered
    last = {(o["method"], o["image_id"]): o for o in outs if o["image_id"] in scenes}
    methods = sorted({k[0] for k in last}, key=lambda x: ORDER.index(x) if x in ORDER else 99)
    common = set.intersection(*[{i for (mm, i) in last if mm == meth} for meth in methods])
    cover = {meth: sum(1 for (mm, _) in last if mm == meth) for meth in methods}
    print("scenes per method:", cover, "| scored (common):", len(common), flush=True)

    item_rows, img_rows = [], []
    for (meth, iid), o in sorted(last.items()):
        if iid not in common:
            continue
        s = scenes[iid]
        pos, neg = set(s.get("pos_x", s["pos"])), set(s.get("neg_x", s["neg"]))
        pos_l, neg_l = set(s["pos"]), set(s["neg"])
        rel = pos & vset
        seen, cats = set(), []
        for it in o["items"]:
            c = m.match(it)
            if c is not None and c in seen:
                continue                                    # same category named twice: keep the first
            if c is not None:
                seen.add(c)
            st = "correct" if c in pos else "wrong" if c in neg else "unverifiable"
            stl = "correct" if c in pos_l else "wrong" if c in neg_l else "unverifiable"
            len_ok = st == "correct" or (c is not None and bool(m.group.get(c, set()) & pos))
            cats.append((c, st, len_ok, stl, it))
        for rank, (c, st, len_ok, stl, it) in enumerate(cats, 1):
            item_rows.append({"method": meth, "image_id": iid, "room": s["room"], "rank": rank, "item": it,
                              "category": m.id2name.get(c) if c is not None else None, "status": st,
                              "lenient_correct": bool(len_ok), "status_lvis_only": stl})
        r = {"method": meth, "image_id": iid, "room": s["room"], "seconds": o.get("seconds", np.nan),
             "n_items": len(cats), "n_relevant_present": len(rel)}
        for K in KS:
            top = cats[:K]
            corr = sum(x[1] == "correct" for x in top)
            wrong = sum(x[1] == "wrong" for x in top)
            r[f"correct@{K}"], r[f"wrong@{K}"] = corr, wrong
            r[f"precision@{K}"] = corr / (corr + wrong) if corr + wrong else np.nan
            lc = sum(x[2] for x in top)
            lw = sum(x[1] == "wrong" and not x[2] for x in top)
            r[f"precision_lenient@{K}"] = lc / (lc + lw) if lc + lw else np.nan
            r[f"verifiable@{K}"] = (corr + wrong) / K
            cl, wl = sum(x[3] == "correct" for x in top), sum(x[3] == "wrong" for x in top)
            r[f"precision_lvis_only@{K}"] = cl / (cl + wl) if cl + wl else np.nan
            r[f"verifiable_lvis_only@{K}"] = (cl + wl) / K
            named = {x[0] for x in top if x[0] is not None}
            named_grp = set().union(*[m.group.get(c, {c}) for c in named]) if named else set()
            r[f"recall@{K}"] = len(rel & named) / len(rel) if rel else np.nan
            r[f"recall_lenient@{K}"] = len(rel & named_grp) / len(rel) if rel else np.nan
        r["hit@3"] = float(any(x[1] == "correct" for x in cats[:3]))
        r["hit_lenient@3"] = float(any(x[2] for x in cats[:3]))
        img_rows.append(r)
    items, imgs = pd.DataFrame(item_rows), pd.DataFrame(img_rows)
    items.to_csv(os.path.join(a.out, "e2_items.csv"), index=False)
    imgs.to_csv(os.path.join(a.out, "e2_per_image.csv"), index=False)

    metrics = ([f"{p}@{K}" for K in KS for p in ("precision", "precision_lenient", "verifiable", "recall",
                                                 "recall_lenient", "precision_lvis_only", "verifiable_lvis_only")]
               + ["hit@3", "hit_lenient@3"])
    summ = []
    for meth in methods:
        d = imgs[imgs.method == meth]
        row = {"method": meth, "scenes": len(d), "median_seconds": float(np.nanmedian(d.seconds)),
               "mean_items": float(d.n_items.mean()), "short_lists(<3 items)": int((d.n_items < 3).sum())}
        for k in metrics:
            row[k] = float(np.nanmean(d[k])) if d[k].notna().any() else np.nan
            row[k + "_n"] = int(d[k].notna().sum())
            row[k + "_ci"] = boot(d[k].values, rng)
        for K in KS:
            c_, w_ = d[f"correct@{K}"].sum(), d[f"wrong@{K}"].sum()
            row[f"pooled_precision@{K}"] = c_ / (c_ + w_) if c_ + w_ else np.nan
        summ.append(row)
    summ = pd.DataFrame(summ)
    summ.to_csv(os.path.join(a.out, "e2_summary.csv"), index=False)

    wide = {k: imgs.pivot(index="image_id", columns="method", values=k) for k in ("hit@3", "recall@6", "precision@3")}
    diffs = []
    for x, y in PAIRS:
        if x in methods and y in methods:
            for k, w in wide.items():
                mean, ci, n = paired(w[x].values.astype(float), w[y].values.astype(float), rng)
                diffs.append({"a": x, "b": y, "metric": k, "a_minus_b": mean, "ci_lo": ci[0], "ci_hi": ci[1], "n": n})
    diffs = pd.DataFrame(diffs, columns=["a", "b", "metric", "a_minus_b", "ci_lo", "ci_hi", "n"])
    diffs.to_csv(os.path.join(a.out, "e2_paired_differences.csv"), index=False)
    by_room = imgs.groupby(["method", "room"]).agg(n=("image_id", "count"), precision3=("precision@3", "mean"),
                                                   recall6=("recall@6", "mean"), hit3=("hit@3", "mean")).reset_index()
    by_room.to_csv(os.path.join(a.out, "e2_by_room.csv"), index=False)
    um = items[items.category.isna()].assign(it=lambda z: z["item"].map(norm_item))
    unmatched = um.groupby("method").it.agg(lambda z: ", ".join(f"{k} ({v})" for k, v in z.value_counts().head(15).items())) \
        if len(um) else pd.Series(dtype=str)
    unmatched.to_csv(os.path.join(a.out, "e2_top_unmatched_items.csv"))

    # blinded audit sheet: normalised items, random method codes, item_uid join key
    rs = random.Random(SEED)
    ids = sorted(common)
    rs.shuffle(ids)
    aud = items[items.image_id.isin(ids[: a.audit_n]) & (items["rank"] <= 6)].copy()
    aud["item_norm"] = aud["item"].map(norm_item)
    aud = aud[aud.item_norm != ""]
    codes_list = [f"M{i + 1}" for i in range(len(methods))]
    rs.shuffle(codes_list)
    codes = dict(zip(methods, codes_list))
    uniq = aud.drop_duplicates(["image_id", "item_norm"])[["image_id", "item_norm"]].reset_index(drop=True)
    uniq = uniq.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    uniq["item_uid"] = [f"U{i:04d}" for i in range(len(uniq))]
    uniq["url"] = uniq.image_id.map(lambda i: scenes[i]["url"])
    sheet = uniq.sort_values(["image_id", "item_uid"])[["item_uid", "image_id", "url", "item_norm"]].rename(columns={"item_norm": "item"})
    sheet["is_it_in_the_photo(Y/N)"] = ""
    sheet["would_a_patient_here_plausibly_want_or_mention_it(Y/N)"] = ""
    sheet.to_csv(os.path.join(a.out, "audit_sheet.csv"), index=False)
    key = aud.merge(uniq[["image_id", "item_norm", "item_uid"]], on=["image_id", "item_norm"])
    key["method_code"] = key.method.map(codes)
    os.makedirs(os.path.join(a.out, "audit_key_private"), exist_ok=True)
    key[["item_uid", "image_id", "item", "method_code", "rank"]].to_csv(
        os.path.join(a.out, "audit_key_private", "audit_key_DO_NOT_OPEN_BEFORE_RATING.csv"), index=False)
    json.dump(codes, open(os.path.join(a.out, "audit_key_private", "method_codes.json"), "w"))

    f = lambda v, ci=None: ("n/a" if v is None or not np.isfinite(v) else f"{v:.3f}") + (
        f" ({ci[0]:.3f}–{ci[1]:.3f})" if ci is not None and np.isfinite(ci[0]) else "")
    rooms = imgs.drop_duplicates("image_id").room.value_counts()
    L = ["# E2: how well does each method name the useful objects in an everyday scene?", "",
         f"{len(common)} scenes scored (LVIS v1 val / COCO 2017), all methods on the same scenes; rooms: "
         + ", ".join(f"{k} {v}" for k, v in rooms.items()) + f". Scenes per method before intersecting: {cover}. "
         f"AAC vocabulary: {len(vset)} LVIS categories"
         + (f" ({', '.join(vocab['missing'])} not in this LVIS release)" if vocab["missing"] else "") + ".", "",
         "## First screen (top 3 items)", "",
         "| Method | precision@3 | pooled | lenient | verifiable@3 | recall@3 | hit@3 | hit lenient | median s/scene | lists < 3 items |",
         "|---|---|---|---|---|---|---|---|---|---|"]
    for _, x in summ.iterrows():
        L.append(f"| {x.method} | {f(x['precision@3'], x['precision@3_ci'])} (n={x['precision@3_n']}) | {f(x['pooled_precision@3'])} | "
                 f"{f(x['precision_lenient@3'])} | {f(x['verifiable@3'])} | {f(x['recall@3'], x['recall@3_ci'])} | "
                 f"{f(x['hit@3'], x['hit@3_ci'])} | {f(x['hit_lenient@3'])} | {x.median_seconds:.2f} | {x['short_lists(<3 items)']} |")
    L += ["", "LVIS-only check (no COCO extension), precision@3 / verifiable@3: "
          + "; ".join(f"{x.method} {f(x['precision_lvis_only@3'])} / {f(x['verifiable_lvis_only@3'])}" for _, x in summ.iterrows()),
          "", "## Deeper lists", "",
          "| Method | precision@6 | recall@6 | recall lenient@6 | precision@12 | recall@12 | verifiable@12 |",
          "|---|---|---|---|---|---|---|"]
    for _, x in summ.iterrows():
        L.append(f"| {x.method} | {f(x['precision@6'])} | {f(x['recall@6'], x['recall@6_ci'])} | {f(x['recall_lenient@6'])} | "
                 f"{f(x['precision@12'])} | {f(x['recall@12'])} | {f(x['verifiable@12'])} |")
    L += ["", "## Paired differences (same scenes; 95% bootstrap CI)", "", "| A − B | metric | difference | n |", "|---|---|---|---|"]
    for _, x in diffs.iterrows():
        L.append(f"| {x.a} − {x.b} | {x.metric} | {f(x.a_minus_b, (x.ci_lo, x.ci_hi))} | {x.n} |")
    L += ["", "## By room", "", "| Method | Room | n | precision@3 | recall@6 | hit@3 |", "|---|---|---|---|---|---|"]
    for _, x in by_room.iterrows():
        L.append(f"| {x.method} | {x.room} | {x.n} | {f(x.precision3)} | {f(x.recall6)} | {f(x.hit3)} |")
    st = items.groupby("method").status.value_counts(normalize=True).unstack().fillna(0)
    L += ["", "## Item status (all ranks)", "", "```", st.round(3).to_string(), "```", "",
          "## Most frequent items that matched no LVIS category", "", "```", unmatched.to_string(), "```", "",
          "## Examples (first 3 audit scenes)", "", "```"]
    for iid in ids[:3]:
        L.append(f"scene {iid} ({scenes[iid]['room']}), AAC objects present: "
                 + ", ".join(m.id2name[c] for c in sorted(set(scenes[iid].get('pos_x', scenes[iid]['pos'])) & vset)))
        for meth in methods:
            it = items[(items.image_id == iid) & (items.method == meth)].sort_values("rank")
            L.append(f"  {meth:11s}: " + ", ".join(f"{r.item}[{r.status[0]}]" for r in it.head(6).itertuples()))
    L += ["```", "", "Status letters: c = correct, w = wrong (verified absent), u = unverifiable.", "",
          "Audit: rate `audit_sheet.csv` (two Y/N columns). The method key is in `audit_key_private/`: "
          "do not open it, or give it to the rater, before rating is finished."]
    open(os.path.join(a.out, "E2_REPORT.md"), "w").write("\n".join(L) + "\n")
    print("\n".join(L))


if __name__ == "__main__":
    main()


## 2. Choose the 300 scenes and download the photos (about 3–5 minutes)
Check the output: the number of scenes per room, and **images downloaded: 300/300** (a few failures are fine).

In [ ]:
!cd /content/e2 && python e2_build_set.py --lvis /content/lvis_v1_val.json --out "{WORK}/set" --per-room 60 --coco-ann /content/instances_train2017.json /content/instances_val2017.json --coco-synset /content/coco_to_synset.json

## 3. Dry run: every method on 3 scenes (about 10–15 minutes, mostly downloading the models)
Each method should print a list of real object names for the first scene. **If any method crashes or prints nonsense, stop and read this cell's output before going on.**

In [ ]:
!cd /content/e2 && python e2_generate.py --set "{WORK}/set" --out /content/e2_dry --limit 3 --methods static clip blip2_orig qwen_vl
import json
for l in open('/content/e2_dry/outputs.dry.jsonl'):
    o = json.loads(l)
    print(f"{o['method']:11s} scene {o['image_id']}: {', '.join(o['items'][:8])}   ({o['seconds']:.2f} s)")

## 4. All 300 scenes (about 30–45 minutes; resumable)

In [ ]:
!cd /content/e2 && python e2_generate.py --set "{WORK}/set" --out "{WORK}/outputs" --methods static clip blip2_orig qwen_vl

## 5. Score and report (1 minute)

In [ ]:
!cd /content/e2 && python e2_score.py --set "{WORK}/set" --outputs "{WORK}/outputs/outputs.jsonl" --out "{WORK}/results"

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{WORK}/results/E2_REPORT.md').read()))

## 6. Download the results
Download `E2_results.zip`. It contains the report, the per-item tables and the blinded audit sheet (`audit_sheet.csv`).

In [ ]:
import shutil, os
os.makedirs('/content/E2_results', exist_ok=True)
shutil.copytree(f'{WORK}/results', '/content/E2_results/results', dirs_exist_ok=True)
shutil.copy(f'{WORK}/outputs/outputs.jsonl', '/content/E2_results/outputs.jsonl')
shutil.copy(f'{WORK}/set/set.jsonl', '/content/E2_results/set.jsonl')
shutil.make_archive('/content/E2_results', 'zip', '/content/E2_results')
from google.colab import files; files.download('/content/E2_results.zip')